# DietBot — Model Training

This notebook trains the two models the app uses:

1. A **Random Forest Regressor** that predicts daily macro targets (protein/carbs/fat/sugar) from height, weight and a calorie target.
2. A **NearestNeighbors** model that matches foods to a macro profile, used to build meal plans.

Running this notebook end-to-end regenerates the `.pkl` files in `../models/`. The app doesn't need you to run this — the trained models are already committed — this is here so the training process is visible and reproducible.

The equivalent of this notebook also exists as a plain script at `../scripts/train_models.py`, which is what CI/automation would actually call.

In [ ]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

DATA_DIR = "../data"
MODELS_DIR = "../models"

## Part 1 — Macro prediction model

In [ ]:
df = pd.read_excel(f"{DATA_DIR}/macro_targets.xlsx", sheet_name="Sheet1")
df.columns = [c.strip() for c in df.columns]
df["Food Energy (Calories/day)"] = (
    df["Food Energy (Calories/day)"].astype(str).str.replace(",", "").astype(float)
)
df.describe()

In [ ]:
INPUT_FEATURES = ["Height", "Weight", "Food Energy (Calories/day)"]
OUTPUT_FEATURES = ["Protein (grams/day)", "Carbs (grams/day)", "Fat (grams/day)", "Sugar (grams/day)"]

df = df.dropna(subset=INPUT_FEATURES + OUTPUT_FEATURES)
print(f"{len(df)} rows after dropping missing values")

X = df[INPUT_FEATURES]
y = df[OUTPUT_FEATURES]

In [ ]:
scaler_x = StandardScaler().fit(X)
scaler_y = StandardScaler().fit(y)
X_scaled = scaler_x.transform(X)
y_scaled = scaler_y.transform(y)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

In [ ]:
macro_model = RandomForestRegressor(random_state=42, n_estimators=100)
macro_model.fit(X_train, y_train)

y_pred = macro_model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

This dataset is small (under 100 training rows after the split), so the R² here is optimistic — it's a demonstration model rather than something clinically validated. See the README's Limitations section.

In [ ]:
def predict_nutrition(height, weight, calories):
    input_df = pd.DataFrame([[height, weight, calories]], columns=INPUT_FEATURES)
    input_scaled = scaler_x.transform(input_df)
    prediction_scaled = macro_model.predict(input_scaled)
    protein, carbs, fat, sugar = scaler_y.inverse_transform(prediction_scaled)[0]
    return {
        "Protein (g/day)": round(protein, 1),
        "Carbs (g/day)": round(carbs, 1),
        "Fat (g/day)": round(fat, 1),
        "Sugar (g/day)": round(sugar, 1),
    }

predict_nutrition(height=182, weight=75, calories=2800)

In [ ]:
joblib.dump(macro_model, f"{MODELS_DIR}/random_forest_regressor.pkl")
joblib.dump(scaler_x, f"{MODELS_DIR}/scaler_X.pkl")
joblib.dump(scaler_y, f"{MODELS_DIR}/scaler_y.pkl")
print("Saved macro model + scalers")

## Part 2 — Food matching model

In [ ]:
food_data = pd.read_csv(f"{DATA_DIR}/food_data.csv", encoding="latin1")

FOOD_COLUMNS = {
    "Food_name": "food_name",
    "Protein(g)": "protein",
    "Total lipid (fat)(g)": "fat",
    "Carbohydrate, by difference(g)": "carbs",
}

foods = (
    food_data[list(FOOD_COLUMNS)]
    .dropna()
    .rename(columns=FOOD_COLUMNS)
    .reset_index(drop=True)
)
print(f"{len(foods)} foods with complete macro data")
foods.head()

In [ ]:
food_matcher = NearestNeighbors(n_neighbors=5, metric="euclidean")
food_matcher.fit(foods[["protein", "fat", "carbs"]].to_numpy())

joblib.dump(food_matcher, f"{MODELS_DIR}/nearest_neighbors_model.pkl")
print("Saved food matcher")

In [ ]:
# Sanity check: what does it recommend for a made-up target?
sample_target = [[40, 15, 50]]  # protein, fat, carbs in grams
distances, indices = food_matcher.kneighbors(sample_target)
foods.iloc[indices[0]]

## Part 3 — A quick look at the data

In [ ]:
import matplotlib.pyplot as plt

bmi_counts = df["bmi_range"].value_counts() if "bmi_range" in df.columns else None
if bmi_counts is not None:
    plt.figure(figsize=(7, 4))
    plt.bar(bmi_counts.index, bmi_counts.values, color="skyblue")
    plt.title("BMI Distribution in the Training Data")
    plt.xlabel("BMI Range")
    plt.ylabel("Number of Records")
    plt.tight_layout()
    plt.show()

In [ ]:
mean_macros = df[["Protein (grams/day)", "Carbs (grams/day)", "Fat (grams/day)"]].mean()

plt.figure(figsize=(7, 4))
plt.bar(mean_macros.index, mean_macros.values, color=["#ff9999", "#66b3ff", "#99ff99"])
plt.title("Average Daily Macro Targets in the Training Data")
plt.ylabel("Grams per day")
plt.tight_layout()
plt.show()

## Notes

- The macro-prediction model only sees three inputs (height, weight, calorie target) — it doesn't know age, gender or sport directly. Those factor into the *calorie target* passed in (see `src/calculations.py`), not into the model itself.
- The food matcher works on raw protein/fat/carb grams, so a food's "distance" from a target is dominated by whichever macro has the largest absolute values (usually carbs). That's a reasonable first pass, not a nutritionally optimal one — see the README's Future Improvements for a better-weighted or optimization-based version.